# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR² colorectal cancer dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. We walk through metadata inspection, recordset enumeration, extraction by Croissant `@id`, filtering, transformation, and basic plotting.

### Dataset Source
The dataset is defined by a Croissant schema and is available here:
<br>https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. This step initializes the Dataset object and allows us to inspect Croissant metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset Croissant metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
# Optionally show further metadata fields
print('Identifier:', metadata.identifier)
print('Publication date:', metadata.date_published)
print('License:', metadata.license)
print('Keywords:', ', '.join(metadata.keywords))

## 2. Data Overview

Review available **record sets**, **fields**, and their `@id` values. All subsequent data references use `@id`s for full traceability as defined in the Croissant schema.

We use the Dataset object methods for listing the available record sets and inspecting their field and column `@id`s.

In [ ]:
# List available record sets with their Croissant @id
recordsets = list(dataset.record_sets())

print('Available Record Sets:')
for rs in recordsets:
    print(f"- Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()

**Display a preview of record data for each available record set, referenced by `@id`.**

In [ ]:
# Display 1-2 records from each record set to preview column names and structure
for rs in recordsets:
    print(f"=== Record Set '{rs.name}' (@id: {rs.id}) ===")
    records = list(dataset.records(record_set=rs.id))
    # Show up to 2 records; these return dicts of {field @id: value}
    for rec in records[:2]:
        pprint.pprint(rec)
    print('-'*50)

## 3. Data Extraction

Load data from each record set into a pandas DataFrame for analysis, referencing each entity by its Croissant `@id`.

We'll build a dictionary `dataframes` mapping each record set's `@id` to its corresponding DataFrame. The column names will be the Croissant field `@id`s.

In [ ]:
# Load all record sets into pandas DataFrames using their @id
dataframes = {}  # map: record_set @id -> pd.DataFrame
for rs in recordsets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded {len(df)} records for record set '{rs.name}' (@id: {rs.id})")
    print(f"Field @id columns: {df.columns.tolist()}\n")
# For further analysis, choose the largest/main record set; here, we select the first by default
MAIN_RECORDSET_ID = recordsets[0].id if recordsets else None
if MAIN_RECORDSET_ID:
    print(f"Example preview for main table: {MAIN_RECORDSET_ID}")
    display(dataframes[MAIN_RECORDSET_ID].head())

## 4. Exploratory Data Analysis (EDA)

Let's demonstrate typical preprocessing using Croissant `@id` as column selectors. We will:
- Select a numeric field (identified by `dataType` in metadata overview)
- Filter on this field, e.g., values exceeding a threshold
- Normalize this numeric field (z-score)
- Optionally, group by a categorical field and compute groupwise means

**All variables use their Croissant-provided `@id` for full traceability.**

In [ ]:
# Identify a numeric field @id and a group-by field @id by inspecting metadata
# Here, we assume the main recordset contains 'dv:age_at_second_crc' (age at 2nd CRC, dataType: Integer)
# and 'dv:sex' (sex group), you may edit to match real @id values from section 2 output

# Set these to the actual @id values present in your data
numeric_field_id = None
group_field_id = None

# Auto-search for a numeric field and a categorical group field
for rs in recordsets:
    if rs.id == MAIN_RECORDSET_ID:
        for field in rs.fields:
            if not numeric_field_id and field.data_type in ["Integer", "Float", "Number"]:
                numeric_field_id = field.id
            if not group_field_id and field.data_type == "Text":
                group_field_id = field.id
        break

print(f"Using numeric field @id: {numeric_field_id}")
print(f"Using group field @id: {group_field_id}")

df = dataframes[MAIN_RECORDSET_ID]
# Drop NA for ease
edf = df.dropna(axis=0, subset=[numeric_field_id])

# Example: filter for values above a threshold
threshold = edf[numeric_field_id].mean()  # mean as example threshold
filtered_df = edf[edf[numeric_field_id] > threshold].copy()

print(f"Filtered records with field '{numeric_field_id}' > {threshold:.1f}:")
display(filtered_df.head())

# Normalize this numeric column (z-score)
col = numeric_field_id + '_normalized'
filtered_df[col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}':")
display(filtered_df[[numeric_field_id, col]].head())

if group_field_id:
    # Group by the group field
    grouped_means = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean')
    print(f"\nGroup-wise mean '{numeric_field_id}' by '{group_field_id}':")
    display(grouped_means)

## 5. Visualization

Visualize the distribution of a numeric field, and optionally, compare group means if a grouping field is available.

Below, all columns are referenced by their Croissant `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], kde=True, bins=12, color='steelblue')
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of '{numeric_field_id}'")
plt.show()

if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"Distribution of '{numeric_field_id}' by '{group_field_id}'")
    plt.show()

## 6. Conclusion

We have successfully explored the FAIR² dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id`. This approach ensures reproducibility and clarity in dataset structure. We loaded all record sets, extracted DataFrames by record set `@id`, and performed basic filtering, normalization, and visualization using metadata-anchored field references.

**Key observations:**
- The main record set provides rich clinicopathological data, including age, sex, and several biomarker fields.
- All analysis steps explicitly use Croissant `@id`, making exploration fully FAIR.
- This notebook template can be adapted for similar Croissant-compliant datasets.